## This is the code to train the model and acquire influence for Number of Samples Experiment

**Default Code**:   
The current default code is a runnable sample. It runs on the synthetic dataset generated with sklearn's make_classification function. The default version code provides the synthetic dataset with 16500 samples and 160 features in total. The separation is set to 5 to make sure the dataset is distinguishable by the model. All features are set to be informative to ensure they are of equal importance. The dataset has only two labels, so it is a binary classification problem. The default setting will then generate the training set and test set from the pool. The default training sample size is 8000, and the test size is 500. The number of features is set to 10. The model in default will be a Simple FeedForward Neural Network constructed by TensorFlow. The Influence Estimation methods we provide by default are the Influence Function and TracIn. If you simply press 'play', the default code will generate ranked influence lists for both Influence Function and TracIn with respect to the above mentioned setting in the root directory. The result lists could then be fed into other analyses.

**By default, this is exactly the same code as the base code. Please refer to the base code for more detailed explanation.**

**Guideline**:  
Read in / Construct Datasets -> **Choose the Training Sample Size** -> Pre-processing -> Model Training -> Influence Estimation -> Store the Ranked Influence lists -> **Change the Training Sample Size and Repeat all the process** -> ... -> **After all the training and estimation, feed the results into the analysis code**  (Remember to change the file name in the last block to save lists in different settings.)

# Import Area

Here is the area to place all the import codes. You don't need to change here unless you want to customise in later sections.

In [105]:
import tensorflow as tf
print(tf.__version__)

2.11.0


In [106]:
import tensorflow as tf
import tensorflow_datasets as tfds
import keras
from keras.utils import to_categorical
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [107]:
from keras import Sequential
from keras.layers import Dense, BatchNormalization, Dropout
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam

In [108]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

In [109]:
import random
from keras.optimizers import SGD

In [110]:
from sklearn.datasets import make_classification

# Dataset Construction Area

**You can use any dataset you wish here, either regression or classification. But in default, since we are using influenciae's IF and TC method, make sure they are split into train and test sets, and then stored as tensorflow dataset format. If you only want to change the dataset, you can only change the code in the first two blocks to read in/ generate your own dataset. But remember to have features X and target y before going to the third block. Also, if you wish to everything on your own, remember to add id inside the dataset.** Since our default code is for classification, the regression might need a lot of changes in all the following sections.


**Input**: Dataset chosen(Usually in Features X and Target y format)  
**Output**: Tensorflow format Train and Test Set  
**Guideline**: Input -> Turn into Dataframe and add ID -> Pre-Processing -> Change the format to Tensorflow -> Output

The default code now produces a synthetic dataset with 16500 pool, 160 features with binary classification problems. The later options will turn that into a 10 features, 8000 train set and 500 test set sample. Both sets will then be turned into TensorFlow format and will wait for training.

**The most important thing in this code is changing the train size. In this experiment, all the other things are fixed, but the number of training sample is changing to test on different number of samples.**

1. Set your default setting here. train_pool + test_size = Total Dataset Size. train_sizes determines the later subset data. Sep to make sure the dataset is distinguishable.

In [111]:
train_pool = 53440
test_size = 500
train_sizes=[53440]
seed=42
ratios = [(9,1), (8,2), (7,3), (6,4), (5,5)]

2. Construct the Synthetic Dataset with Make Classification here. **Could replace this with other datasets with X and y.**

In [112]:
df = pd.read_csv("diamonds.csv")

In [113]:
median_price = df["price"].median()
df["label"] = (df["price"] > median_price).astype(int)
df = df.drop(columns=['price'])

3. Turn the X and y into dataframe for easy further processing. Add ID column to easy retrieve samples within IF/TC.

In [114]:
df['id'] = np.arange(1, len(df) + 1)
print(df)

       features/carat  features/clarity  features/color  features/cut  \
0                1.26                 2               4             2   
1                0.80                 3               4             4   
2                0.56                 4               2             4   
3                1.51                 3               6             1   
4                0.33                 6               5             4   
...               ...               ...             ...           ...   
53935            1.02                 2               4             2   
53936            0.93                 2               4             3   
53937            0.30                 4               5             4   
53938            0.36                 3               2             4   
53939            0.70                 1               2             2   

       features/depth  features/table  features/x  features/y  features/z  \
0                60.6            60.0        6

In [115]:
df_train_pool = df.iloc[:train_pool].reset_index(drop=True)
df_test = df.iloc[train_pool:].reset_index(drop=True)

In [116]:
print(df_train_pool.head())
print(df_test.head())

   features/carat  features/clarity  features/color  features/cut  \
0            1.26                 2               4             2   
1            0.80                 3               4             4   
2            0.56                 4               2             4   
3            1.51                 3               6             1   
4            0.33                 6               5             4   

   features/depth  features/table  features/x  features/y  features/z  label  \
0            60.6            60.0        6.97        7.00        4.23      1   
1            62.1            54.0        5.96        5.99        3.71      1   
2            61.7            54.0        5.28        5.32        3.27      0   
3            64.0            58.0        7.24        7.27        4.64      1   
4            62.2            54.0        4.43        4.45        2.76      0   

   id  
0   1  
1   2  
2   3  
3   4  
4   5  
   features/carat  features/clarity  features/color  fea

4. Here, we choose the subset of the full dataset. By setting features_to_test, we have the subset feature size. By changing nested_train_dfs, we have different sample sizes.

In [117]:
nested_train_dfs = [df_train_pool.iloc[:size].reset_index(drop=True) for size in train_sizes]

**The most important thing in this experiment is the following code**:  
Based on the train_sizes=[1000,2000,3000,4000,5000,6000,7000,8000,9000,10000] defined above, we could map the number of training samples with the following code. By choosing the number in [], we could modify the train set size. Therefore, only changing the following code block is enough to produce the experiment result successfully.

In [118]:
train_df = nested_train_dfs[0]

In [119]:
train_df["clean_label"] = train_df["label"].copy()

# Randomly select 20% of the training samples
noise_ratio = 0.20
noise_seed = 42

rng = np.random.default_rng(noise_seed)

n_noisy = int(len(train_df) * noise_ratio)

noisy_positions = rng.choice(
    len(train_df),
    size=n_noisy,
    replace=False
)

# Indicator showing which samples were corrupted
train_df["is_noisy"] = 0
train_df.loc[train_df.index[noisy_positions], "is_noisy"] = 1

# Flip the binary labels: 0 -> 1 and 1 -> 0
train_df.loc[
    train_df.index[noisy_positions],
    "label"
] = 1 - train_df.loc[
    train_df.index[noisy_positions],
    "label"
]

# Save a clearer name for the labels used during training
train_df["noisy_label"] = train_df["label"]

print("Number of training samples:", len(train_df))
print("Number of flipped labels:", train_df["is_noisy"].sum())
print("Noise ratio:", train_df["is_noisy"].mean())

print(
    train_df[
        ["id", "clean_label", "noisy_label", "is_noisy"]
    ].head()
)

Number of training samples: 53440
Number of flipped labels: 10688
Noise ratio: 0.2
   id  clean_label  noisy_label  is_noisy
0   1            1            0         1
1   2            1            1         0
2   3            0            0         0
3   4            1            1         0
4   5            0            0         0


In [120]:
# Feature columns used by the model
selected_features = [
    col for col in train_df.columns
    if col.startswith("features/")
]

# Preserve IDs separately
train_ids_original = train_df["id"].to_numpy()

# Scale IDs only for the influence pipeline
IDs = (
    train_ids_original
    .reshape(-1, 1)
    .astype(np.float32)
    / 1e10
)

# Model features
X_train_features = train_df[
    selected_features
].to_numpy(dtype=np.float32)

# Append the ID column, as required by your existing influence code
X_train = np.hstack([
    X_train_features,
    IDs
])

# Use the corrupted labels for training
y_train_1d = train_df[
    "noisy_label"
].to_numpy(dtype=np.int64)

y_train = to_categorical(
    y_train_1d,
    num_classes=2
)

print(X_train.shape)
print(y_train.shape)

(53440, 10)
(53440, 2)


In [121]:
# X_train = train_df.drop(columns=["label"])
# y_train = train_df["label"]
# IDs = X_train["id"].values.reshape(-1, 1).astype(np.float32)
# IDs = IDs  / 1e10

# X_train = X_train.drop(columns=["id"]).values.astype(np.float32)
# X_train = np.hstack((X_train, IDs))
# y_train = to_categorical(y_train.values,num_classes=2)

# print(X_train)

In [122]:
test_df = df_test
X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]
IDs = X_test["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_test = X_test.drop(columns=["id"]).values.astype(np.float32)
X_test = np.hstack((X_test, IDs))
y_test = to_categorical(y_test.values,num_classes=2)

print(X_test.shape)
print(y_test.shape)

(500, 10)
(500, 2)


5 (Optional) The following code below can display the samples distribution. Uncomment them to acquire the distribution plot.

In [123]:
# X_all = np.vstack([X_train, X_test])

# y_train_1d = np.argmax(y_train, axis=1)
# y_test_1d  = np.argmax(y_test, axis=1)
# y_all_1d = np.hstack([y_train_1d, y_test_1d])

In [124]:
# from sklearn.metrics import pairwise_distances
# from sklearn.manifold import MDS
# import seaborn as sns
# import matplotlib.pyplot as plt

In [125]:
# D = pairwise_distances(X_all) 

In [126]:
# X_mds = MDS(n_components=2, dissimilarity='precomputed', random_state=0).fit_transform(D)

In [127]:
# sns.scatterplot(x=X_mds[:,0], y=X_mds[:,1], hue=y_all_1d)
# plt.title("MDS – preserves original distances")

6. Now we have the train_ds and test_ds for training

In [128]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Training Area

**Could modify the model as you wish here. Again, in default, influenciae relies on TensorFlow, so use the TensorFlow model if you only want to change the model. Remember: Store the InfluenceModel into the model_list with the loss function. The InfluenceModel will be used to obtain influence later. If you don't change the estimation methods, then the final output at this step shall always be the model_list**

**Input**:Train and Test Set from Data Construction Section   
**Output**: Model List  
**Guideline**: Input -> Define the Model and Hyperparameters -> Train the Model -> Output

Always remember to train the model, get the influence model and store that in model list, unless you wish to change the estimation methods.

The default code now use the train and test set generated from the last section to train the model. The default hyperparameters are: 300 Epochs, Simple FeedForward Neural Network, CategoricalCrossEntropy Loss function, SGD optimizer. Within each epoch, the current model will be turned into an Influence Model and stored inside a model list. After the training, the model list will be passed to next section for influence estimation.

In [129]:
from tensorflow.keras.regularizers import l2

1. **Could modify the model as you wish here as long as it is tensorflow.** Just remember: Store the InfluenceModel into the model_list with the loss function

In [130]:
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  
    BatchNormalization(momentum=0.9),
    Dropout(0.0),
    Dense(8, activation='relu'),
    Dense(y_train.shape[1])
])
loss_fn = CategoricalCrossentropy(from_logits=True)
optimizer = SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss=loss_fn, optimizer=optimizer, metrics=['accuracy'])

epochs = 44
unreduced_loss_fn = CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
model_list = []
initial_model = tf.keras.models.clone_model(model)
initial_model.set_weights(model.get_weights())
model_list.append(InfluenceModel(initial_model, start_layer=-1, loss_function=unreduced_loss_fn))
for i in range(epochs):
  model.fit(train_ds.batch(512), epochs=1, validation_data=test_ds.batch(512), verbose=2)
  checkpoint_model = tf.keras.models.clone_model(model)
  checkpoint_model.set_weights(model.get_weights())
  model_list.append(InfluenceModel(checkpoint_model, start_layer=-1, loss_function=unreduced_loss_fn))
base_loss, acc = model.evaluate(test_ds.batch(64), verbose=2)
print(base_loss)

105/105 - 1s - loss: 0.6754 - accuracy: 0.6102 - val_loss: 0.4765 - val_accuracy: 0.8380 - 1s/epoch - 11ms/step
105/105 - 0s - loss: 0.5742 - accuracy: 0.7302 - val_loss: 0.4013 - val_accuracy: 0.8900 - 497ms/epoch - 5ms/step
105/105 - 0s - loss: 0.5589 - accuracy: 0.7565 - val_loss: 0.3719 - val_accuracy: 0.9340 - 485ms/epoch - 5ms/step
105/105 - 0s - loss: 0.5533 - accuracy: 0.7668 - val_loss: 0.3585 - val_accuracy: 0.9420 - 490ms/epoch - 5ms/step
105/105 - 0s - loss: 0.5503 - accuracy: 0.7709 - val_loss: 0.3517 - val_accuracy: 0.9460 - 486ms/epoch - 5ms/step
105/105 - 0s - loss: 0.5483 - accuracy: 0.7738 - val_loss: 0.3473 - val_accuracy: 0.9460 - 495ms/epoch - 5ms/step
105/105 - 0s - loss: 0.5467 - accuracy: 0.7752 - val_loss: 0.3392 - val_accuracy: 0.9500 - 494ms/epoch - 5ms/step
105/105 - 0s - loss: 0.5454 - accuracy: 0.7757 - val_loss: 0.3350 - val_accuracy: 0.9500 - 498ms/epoch - 5ms/step
105/105 - 0s - loss: 0.5444 - accuracy: 0.7764 - val_loss: 0.3346 - val_accuracy: 0.9500 -

In [131]:
train_logits = model.predict(
    X_train,
    batch_size=256,
    verbose=0
)

# Calculate one loss value per sample using the corrupted labels
per_sample_loss_fn = CategoricalCrossentropy(
    from_logits=True,
    reduction=tf.keras.losses.Reduction.NONE
)

training_losses = per_sample_loss_fn(
    y_train,
    train_logits
).numpy()

print(training_losses.shape)
print(pd.Series(training_losses).describe())

(53440,)
count    53440.000000
mean         0.530147
std          0.513557
min          0.000000
25%          0.207925
50%          0.282884
75%          0.602367
max          5.164397
dtype: float64


In [132]:
noise_loss_df = pd.DataFrame({
    "Train_ID": train_ids_original,
    "Clean_Label": train_df["clean_label"].to_numpy(),
    "Noisy_Label": train_df["noisy_label"].to_numpy(),
    "is_noisy": train_df["is_noisy"].to_numpy(),
    "Training_Loss": training_losses
})

print(noise_loss_df.head())
print(
    noise_loss_df.groupby("is_noisy")[
        "Training_Loss"
    ].describe()
)

   Train_ID  Clean_Label  Noisy_Label  is_noisy  Training_Loss
0         1            1            0         1       1.639773
1         2            1            1         0       0.560392
2         3            0            0         0       0.403948
3         4            1            1         0       0.175827
4         5            0            0         0       0.226923
            count      mean       std       min       25%       50%       75%  \
is_noisy                                                                        
0         42752.0  0.299664  0.165031  0.000000  0.196040  0.247026  0.334866   
1         10688.0  1.452068  0.383833  0.026263  1.238864  1.508526  1.724322   

               max  
is_noisy            
0         5.164397  
1         3.800088  


In [133]:
noise_loss_df.to_csv(
    "Noise_GroundTruth_and_TrainingLoss.csv",
    index=False
)

In [134]:
train_df.to_csv(
    "NoisyLabel_TrainingData.csv",
    index=False
)

test_df.to_csv(
    "Clean_TestData.csv",
    index=False
)

# Influence Estimation Area

**Again, you could use other influence analysis methods rather than IF/TC. You can also use any other Influence Function or TracIn implementation. Just Remember: 1. Make sure the package is unform throughout the framework. 2. Generate a Ranked influence list for each Influence Function and TracIn; Only the ranked influence list could be fed into the following analysis code.**

**The default code now use the model list, train set and test set to estimate the influence, and produce a ranked influence list for both IF and TC. The results are then saved in the root directory.**

**Input**:Model list from Training section, Train and Test Set from Data Construction Section   
**Output**: Two ranked Influence Lists for IF and TC.  
**Guideline**: Input -> Influence Estimation Methods -> Influence Matrix -> Output

1. Influence Function: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use influence_matrix.

In [135]:
from tqdm import tqdm

In [136]:
train_ids = []
test_ids = []
train_samples_np = np.array([x.numpy() for x, y in train_ds])
train_ids = [round(sample[-1] * 1e10) for sample in train_samples_np]

In [137]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

influence_model = model_list[-1]
ihvp_calculator = ExactIHVP(influence_model, train_ds.batch(64))
influence_calculator = FirstOrderInfluenceCalculator(influence_model, train_ds, ihvp_calculator)

influence_matrix = np.zeros((num_test_samples, num_train_samples))

samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(64), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in tqdm(enumerate(explanation_ds.as_numpy_iterator()),total=num_test_samples,desc="Computing influence"):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            influence_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(influence_matrix, axis=0).reshape(1, -1)
df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(df)

Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089


Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089
2026-07-19 12:50:11.105094: I tensorflow/core/util/cuda_solvers.cc:179] Creating GpuSolver handles for stream 0x2b4ee870
Computing influence: 100%|██████████| 500/500 [08:00<00:00,  1.04it/s]


       Train_ID     Score
0             1 -0.798092
1             2  0.224097
2             3  0.153661
3             4  0.206177
4             5  0.039817
...         ...       ...
53435     53436  0.195223
53436     53437  0.117805
53437     53438  0.214631
53438     53439  0.213365
53439     53440  0.195968

[53440 rows x 2 columns]


2. TracIn: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use TracIn_matrix.

In [138]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

TracIn_matrix = np.zeros((num_test_samples, num_train_samples))
influence_calculator = TracIn(
    model_list, 0.001
)
samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(64), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in tqdm(enumerate(explanation_ds.as_numpy_iterator()),total=num_test_samples,desc="Computing influence"):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            TracIn_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(TracIn_matrix, axis=0).reshape(1, -1)
TracIn_df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(TracIn_df)

Computing influence: 100%|██████████| 500/500 [09:30<00:00,  1.14s/it]


       Train_ID     Score
0             1  3.234391
1             2 -0.008772
2             3  3.306922
3             4 -0.002540
4             5  3.411973
...         ...       ...
53435     53436  3.457626
53436     53437  3.449038
53437     53438  3.406805
53438     53439  3.411435
53439     53440 -0.002004

[53440 rows x 2 columns]


3. Here we turn both influence lists to the ranked influence lists and then store them for further processing.

In [139]:
df_sorted = df.sort_values(by="Score", ascending=False).reset_index(drop=True)

TracIn_sorted = TracIn_df.sort_values(by="Score", ascending=False).reset_index(drop=True)

In [140]:
TracIn_sorted.to_csv("NoisyLabel_TracIn_Scores.csv",index = False)
df_sorted.to_csv("NoisyLabel_FOIF_Scores.csv",index = False)